In [1]:
import os, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from adaptive_behaviour import paths

import numpy as np
import pandas as pd

TRIALS   = paths.RUNNERS
SOURCE   = paths.RUNNERS / "trial_si.py"       # the published AIF-1 runner
PATCHED  = paths.RUNNERS_GENERATED / "trial_si_uniform.py"   # generated below
OUT_DIR  = paths.RESULTS_AIF_PRIOR             # per-seed results land here
PUBLISHED = paths.RESULTS_SI / "p_data.csv"            # existing AIF-1 results

N_SEEDS   = 12    # raise this and re-run; completed seeds are skipped
N_WORKERS = 6     # of 12 cores; leave headroom so the machine stays usable
SEEDS     = list(range(N_SEEDS))

OUT_DIR.mkdir(exist_ok=True)
assert SOURCE.exists(), f"published AIF-1 runner not found at {SOURCE}"
assert PUBLISHED.exists(), "published AIF-1 results not found"
print(f"python   : {paths.rel(sys.executable)}")
print(f"source   : {SOURCE.name}")
print(f"outputs  : {paths.rel(OUT_DIR)}")
print(f"seeds    : {SEEDS[0]}-{SEEDS[-1]} ({N_SEEDS} total), {N_WORKERS} workers")

# Pin BLAS to one thread per worker. Each runner otherwise spawns one BLAS thread per core,
# so N concurrent runners create N x 12 threads on a 12-core machine. With 2432-state matrices
# threaded BLAS gives no speed-up, and the contention dominated everything on the first attempt
# (load average 55 on 12 cores).
BLAS_ENV = dict(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
                VECLIB_MAXIMUM_THREADS="1", NUMEXPR_NUM_THREADS="1")


python   : <repo>/.venv/bin/python
source   : trial_si.py
outputs  : <repo>/main/pong_trials/aif_prior_check
seeds    : 0-11 (12 total), 6 workers


In [2]:
src = SOURCE.read_text()

patches = [
    ("C = random_single_categorical(num_obs)",    "C = obj_array_uniform(num_obs)"),
    ("D = random_single_categorical(num_states)", "D = obj_array_uniform(num_states)"),
    ("OUT_DIR = paths.RESULTS_SI_RAW",            "OUT_DIR = paths.RESULTS_AIF_PRIOR"),
]

out = src
for old, new in patches:
    n = out.count(old)
    assert n >= 1, f"PATCH FAILED - not found in trial_si.py: {old!r}"
    out = out.replace(old, new)
    print(f"  {n}x  {old}\n       -> {new}")

PATCHED.write_text(out)
print(f"\nwrote {PATCHED.name} ({len(out.splitlines())} lines)")

# show the diff so the change is visible in the notebook record
import difflib
diff = list(difflib.unified_diff(src.splitlines(), out.splitlines(),
                                 SOURCE.name, PATCHED.name, lineterm="", n=1))
print("\n".join(diff))

  1x  C = random_single_categorical(num_obs)
       -> C = obj_array_uniform(num_obs)
  1x  D = random_single_categorical(num_states)
       -> D = obj_array_uniform(num_states)
  4x  str('data_n_plot_si/')
       -> str('aif_prior_check/')

wrote trial_si_uniform.py (235 lines)
--- trial_si.py
+++ trial_si_uniform.py
@@ -101,5 +101,5 @@
 
-C = random_single_categorical(num_obs)
-
-D = random_single_categorical(num_states)
+C = obj_array_uniform(num_obs)
+
+D = obj_array_uniform(num_states)
 
@@ -226,6 +226,6 @@
 
-file1 = str('data_n_plot_si/') + str('data_') + str(mtrial) + '.csv'
-file2 = str('data_n_plot_si/') + str('a_data_') +  str(mtrial) + '.csv'
-file3 = str('data_n_plot_si/') + str('b_data_') +  str(mtrial) + '.csv'  
-file4 = str('data_n_plot_si/') + str('c_data_') +  str(mtrial) + '.csv'
+file1 = str('aif_prior_check/') + str('data_') + str(mtrial) + '.csv'
+file2 = str('aif_prior_check/') + str('a_data_') +  str(mtrial) + '.csv'
+file3 = str('aif_prior_check/') + str('b_da

In [3]:
def seed_done(seed):
    return (OUT_DIR / f"data_{seed}.csv").exists()


def run_seed(seed):
    """Run one seed in a subprocess. Returns (seed, elapsed_seconds, returncode, stderr_tail)."""
    t0 = time.time()
    r = subprocess.run([sys.executable, str(PATCHED), str(seed)],
                       capture_output=True, text=True, cwd=str(TRIALS),
                       env=dict(os.environ, **BLAS_ENV))
    return seed, time.time() - t0, r.returncode, r.stderr[-400:] if r.returncode else ""


if seed_done(SEEDS[0]):
    print(f"seed {SEEDS[0]} already complete - skipping timing run")
    per_seed_min = None
else:
    print(f"running seed {SEEDS[0]} alone to measure cost...")
    s, dt, rc, err = run_seed(SEEDS[0])
    per_seed_min = dt / 60
    print(f"seed {s}: {'OK' if rc == 0 else 'FAILED'} in {per_seed_min:.1f} min")
    if rc:
        print(err)
    else:
        remaining = len([x for x in SEEDS if not seed_done(x)])
        rounds = -(-remaining // N_WORKERS)
        print(f"\n{remaining} seeds remain -> ~{rounds} round(s) of {N_WORKERS}")
        print(f"estimated wall-clock: {rounds * per_seed_min:.0f}-{rounds * per_seed_min * 2:.0f} min "
              f"(the range allows for contention between workers)")

running seed 0 alone to measure cost...
seed 0: OK in 48.5 min

11 seeds remain -> ~2 round(s) of 6
estimated wall-clock: 97-194 min (the range allows for contention between workers)


In [4]:
pending = [s for s in SEEDS if not seed_done(s)]
print(f"{len(SEEDS) - len(pending)} of {len(SEEDS)} already complete; running {len(pending)}\n")

failures = []
if pending:
    t_start = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(run_seed, s): s for s in pending}
        for i, fut in enumerate(as_completed(futures), 1):
            seed, dt, rc, err = fut.result()
            status = "OK" if rc == 0 else "FAILED"
            print(f"[{i:>2}/{len(pending)}] seed {seed:>2}  {status:<6} {dt/60:5.1f} min"
                  f"   (elapsed {(time.time()-t_start)/60:5.1f} min)")
            if rc:
                failures.append((seed, err))
    print(f"\ntotal wall-clock: {(time.time()-t_start)/60:.1f} min")

done = [s for s in SEEDS if seed_done(s)]
print(f"seeds complete: {len(done)}/{len(SEEDS)}")
if failures:
    print(f"\n{len(failures)} FAILED:")
    for s, e in failures:
        print(f"  seed {s}: {e[:200]}")

1 of 12 already complete; running 11

[ 1/11] seed  2  OK     140.7 min   (elapsed 140.7 min)
[ 2/11] seed  4  OK     150.3 min   (elapsed 150.3 min)
[ 3/11] seed  5  OK     165.9 min   (elapsed 165.9 min)
[ 4/11] seed  6  OK     186.8 min   (elapsed 186.8 min)
[ 5/11] seed  1  OK     194.5 min   (elapsed 194.5 min)
[ 6/11] seed  3  OK     196.2 min   (elapsed 196.2 min)
[ 7/11] seed  7  OK     164.6 min   (elapsed 305.3 min)
[ 8/11] seed  8  OK     160.4 min   (elapsed 310.8 min)
[ 9/11] seed  9  OK     160.8 min   (elapsed 326.7 min)
[10/11] seed 10  OK     140.9 min   (elapsed 327.7 min)
[11/11] seed 11  OK     139.4 min   (elapsed 333.9 min)

total wall-clock: 333.9 min
seeds complete: 12/12


In [5]:
def load_run(paths, label):
    frames = []
    for p in paths:
        d = pd.read_csv(p)
        frames.append(d)
    d = pd.concat(frames, ignore_index=True)
    d["half"] = d.half.astype(int)
    g = (d.groupby(["session_num", "half"]).hit_count.mean()
           .unstack().rename(columns={0: "w1", 1: "w2"}))
    g["run"] = label
    return g.reset_index()


done = [s for s in SEEDS if seed_done(s)]
assert done, "no completed seeds yet - run the cells above first"

uniform = load_run([OUT_DIR / f"data_{s}.csv" for s in done], "uniform C,D")

pub = pd.read_csv(PUBLISHED)
pub["half"] = pub.half.astype(int)
pub = pub[pub.session_num.isin(done)]
published = (pub.groupby(["session_num", "half"]).hit_count.mean()
               .unstack().rename(columns={0: "w1", 1: "w2"}).reset_index())
published["run"] = "random C,D (published)"

both = pd.concat([published, uniform], ignore_index=True)
both["gain"] = both.w2 - both.w1
both["RI"] = np.where(both.w1 != 0, (both.w2 - both.w1) / both.w1 * 100, np.nan)

summary = both.groupby("run")[["w1", "w2", "gain", "RI"]].mean().round(4)
print(f"paired on {len(done)} seeds: {done}\n")
print(summary.to_string())

paired on 12 seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

half                        w1      w2    gain      RI
run                                                   
random C,D (published)  0.9368  0.7596 -0.1772 -8.5900
uniform C,D             0.8259  0.7035 -0.1225 -7.5467


In [6]:
from scipy import stats

m = published.merge(uniform, on="session_num", suffixes=("_pub", "_uni"))
delta = m.w2_uni - m.w2_pub          # positive => uniform priors help

t, p = stats.ttest_rel(m.w2_uni, m.w2_pub)
w = stats.wilcoxon(m.w2_uni, m.w2_pub) if len(m) >= 6 else None

print(f"Window-2 rally length, paired difference (uniform - published)")
print(f"  n seeds          : {len(m)}")
print(f"  mean difference  : {delta.mean():+.4f}")
print(f"  95% CI           : [{delta.mean() - 1.96*delta.sem():+.4f}, "
      f"{delta.mean() + 1.96*delta.sem():+.4f}]")
print(f"  paired t-test    : t = {t:+.3f}, p = {p:.4f}")
if w is not None:
    print(f"  Wilcoxon         : W = {w.statistic:.1f}, p = {w.pvalue:.4f}")
print(f"\n  published mean   : {m.w2_pub.mean():.4f}")
print(f"  uniform mean     : {m.w2_uni.mean():.4f}")

Window-2 rally length, paired difference (uniform - published)
  n seeds          : 12
  mean difference  : -0.0561
  95% CI           : [-0.1772, +0.0649]
  paired t-test    : t = -0.909, p = 0.3829
  Wilcoxon         : W = 31.0, p = 0.5693

  published mean   : 0.7596
  uniform mean     : 0.7035


In [7]:
MCC_W2, HCC_W2 = 0.847, 0.880
PUB_W2 = 0.801

uni_w2 = m.w2_uni.mean()
shift = uni_w2 - m.w2_pub.mean()

print(f"published AIF-1 (all 99 seeds) : {PUB_W2:.3f}")
print(f"published AIF-1 (these seeds)  : {m.w2_pub.mean():.3f}")
print(f"uniform priors  (these seeds)  : {uni_w2:.3f}")
print(f"shift                          : {shift:+.3f}")
print(f"\nreference: MCC {MCC_W2:.3f}, HCC {HCC_W2:.3f}")

material = (p < 0.05) and (abs(shift) > 0.05)
crosses = uni_w2 > MCC_W2

print("\n" + "=" * 62)
if material and crosses:
    print("MATERIAL - and it changes AIF-1's standing against the cultures.")
    print("=> Run all 99 seeds with uniform priors and regenerate Table 2,")
    print("   Supplementary S1-S4, and the AIF-1 claims in the Results.")
elif material:
    print("MATERIAL shift, but AIF-1 remains below the cultures.")
    print("=> Full re-run still advisable; the AIF-1 conclusion likely survives")
    print("   but the reported numbers would change.")
else:
    print("NOT MATERIAL at this sample size.")
    print("=> Disclose the initialisation mismatch in the Methods parameter")
    print("   table and leave the published AIF-1 results standing.")
print("=" * 62)
print(f"\n(decision rule: p < 0.05 and |shift| > 0.05 rally length; "
      f"here p = {p:.4f}, |shift| = {abs(shift):.3f})")

published AIF-1 (all 99 seeds) : 0.801
published AIF-1 (these seeds)  : 0.760
uniform priors  (these seeds)  : 0.703
shift                          : -0.056

reference: MCC 0.847, HCC 0.880

NOT MATERIAL at this sample size.
=> Disclose the initialisation mismatch in the Methods parameter
   table and leave the published AIF-1 results standing.

(decision rule: p < 0.05 and |shift| > 0.05 rally length; here p = 0.3829, |shift| = 0.056)
